# Week 3: ML on Databricks - Spark & Regression

## Learning Objectives

By the end of this 2-hour session, you will be able to:

- **Understand distributed processing**: Recognize when and why Spark is needed for big data
- **Work with Spark DataFrames**: Perform data operations on large-scale datasets
- **Build feature pipelines**: Engineer ML-ready features using Spark ML transformers
- **Train regression models**: Use Spark MLlib for scalable machine learning
- **Evaluate and compare models**: Select the best regression algorithm for your use case
- **Apply best practices**: Cache data, optimize partitions, and build production-ready ML pipelines

## Prerequisites

Before starting this lab, you should have:

- ✅ Completed Week 1-2 (PyTorch basics, neural networks)
- ✅ Watched pre-class videos on Databricks workspace, Spark DataFrames, and Spark ML basics
- ✅ Access to your cohort's Databricks cluster
- ✅ Basic understanding of regression concepts (covered in pre-class videos)

## Session Format

This is a **hands-on lab session**. The structure for each topic is:

1. **Theory recap** (brief - you watched the videos)
2. **Live demo** (instructor demonstrates the concept)
3. **Lab exercise** (you practice independently)
4. **Verification** (check your work)

## Environment

- **Platform**: Azure Databricks
- **Runtime**: Databricks Runtime 17.3 LTS ML (Python 3.12.3, Spark 4.0.0)
- **Cluster Type**: Shared (Unity Catalog) - Provides secure multi-user isolation
- **Cluster**: Shared cluster with Fair Scheduler (20 students can work concurrently)
- **Dataset**: NYC Taxi Trip Data (1M rows) from Unity Catalog Volume

**Important Notes**:
- Shared clusters restrict `spark.sparkContext` access for security (by design)
- All operations use DataFrame API (RDD operations not supported on Shared clusters)
- This is the recommended configuration for secure multi-user environments

Let's begin!

## Section 0: Environment Setup

Let's start by verifying our Databricks environment and understanding the cluster configuration.

In [ ]:
# No package installation needed - Databricks Runtime 17.3 LTS ML includes all required libraries
# Pre-installed: PySpark, Spark MLlib, pandas, numpy, matplotlib, seaborn, scikit-learn, PyTorch

# Import all necessary libraries
import sys
import pyspark
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.regression import LinearRegression, RandomForestRegressor, GBTRegressor, DecisionTreeRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Verify Python and Spark versions
print("=" * 70)
print("ENVIRONMENT VERIFICATION")
print("=" * 70)
print(f"\nPython version: {sys.version}")
print(f"Spark version: {spark.version}")
print(f"PySpark version: {pyspark.__version__}")

# Show cluster configuration
# Note: Shared clusters restrict direct SparkContext access for security
# This is by design to provide user isolation in multi-tenant environments
print(f"\n--- Cluster Configuration ---")

# Check cluster configuration (Shared clusters restrict some config access)
config_checks = {
    'spark.scheduler.mode': 'Scheduler Mode',
    'spark.sql.adaptive.enabled': 'Adaptive Query Execution',
    'spark.sql.shuffle.partitions': 'Shuffle Partitions',
    'spark.driver.memory': 'Driver Memory',
    'spark.executor.memory': 'Executor Memory'
}

for config_key, display_name in config_checks.items():
    try:
        value = spark.conf.get(config_key)
        print(f"  {display_name}: {value}")
    except Exception:
        print(f"  {display_name}: Managed by cluster (not user-accessible)")

print(f"\n--- Cluster Type ---")
print(f"  Type: Shared (Unity Catalog)")
print(f"  Access Mode: Shared with user isolation")
print(f"  ℹ️  SparkContext access restricted for security")
print(f"  ℹ️  All operations use DataFrame API")

print("\n" + "=" * 70)
print("✅ Environment setup complete!")
print("   Runtime: Databricks ML LTS 17.3")
print("   Spark: 4.0.0 (DataFrame API fully supported)")
print("=" * 70)

## Section 0.5: Dataset Setup

**INSTRUCTOR/FIRST USER: Run this section ONCE before class starts** (5 minutes before students arrive)

This will download the NYC Taxi dataset to the shared Unity Catalog Volume so all students can access it.

In [ ]:
# DATASET SETUP - Run this ONCE before class
# Downloads NYC Taxi dataset to shared DBFS storage

import urllib.request
import os

# DBFS path for shared datasets (works on all cluster types)
SHARED_DATA_PATH = "/dbfs/shared_datasets"
WEEK_03_DATASET = f"{SHARED_DATA_PATH}/week_03_datasets/yellow_tripdata_2024-01.parquet"

print("🔧 Week 3 Dataset Setup")
print("="*70)

try:
    # Check if dataset already exists
    existing_df = spark.read.parquet(WEEK_03_DATASET)
    existing_count = existing_df.count()
    print(f"✅ Dataset already available: {existing_count:,} rows")
    print(f"   Path: {WEEK_03_DATASET}")
    print(f"   Students can proceed to data loading section")
    
    # Pre-cache for students
    existing_df.cache()
    existing_df.count()
    print(f"✅ Dataset cached on cluster for fast access")
    
except Exception:
    # Download dataset
    print("📥 Downloading NYC Taxi dataset from official NYC TLC source...")
    print("   URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet")
    print("   Size: ~45 MB (3 million rows)")
    print("   Time: ~30-90 seconds depending on network")
    print("")
    
    try:
        # Download to temp
        local_temp = "/tmp/taxi_setup.parquet"
        print("   ⏳ Downloading...")
        
        urllib.request.urlretrieve(
            "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet",
            local_temp
        )
        
        file_size_mb = os.path.getsize(local_temp) / (1024 ** 2)
        print(f"   ✅ Downloaded: {file_size_mb:.1f} MB")
        
        # Copy to DBFS
        dbutils.fs.mkdirs(f"{SHARED_DATA_PATH}/week_03_datasets/")
        dbutils.fs.cp(f"file:{local_temp}", WEEK_03_DATASET)
        print(f"   ✅ Saved to shared DBFS: {WEEK_03_DATASET}")
        
        # Verify and cache
        verify_df = spark.read.parquet(WEEK_03_DATASET)
        verify_count = verify_df.count()
        verify_cols = len(verify_df.columns)
        
        verify_df.cache()
        verify_df.count()  # Trigger caching
        
        print(f"   ✅ Verified: {verify_count:,} rows, {verify_cols} columns")
        print(f"   ✅ Cached on cluster for fast student access")
        
        # Cleanup
        os.remove(local_temp)
        print("")
        print("="*70)
        print("🎉 Setup Complete! Students can now proceed.")
        print("="*70)
        
    except Exception as e:
        print(f"\n❌ Setup failed: {str(e)}")
        print("\nTroubleshooting:")
        print("  1. Check internet connectivity")
        print("  2. Verify DBFS permissions (DS_Academy group)")
        print("  3. Ensure DBFS path is accessible: /dbfs/shared_datasets")
        print("  4. Try running validation notebook first (Test 9)")
        raise

## What Are We Building Today?

### The Business Problem: NYC Taxi Fare Prediction at Scale

**Scenario:** You're a data scientist at a ride-sharing analytics company. Your team needs to build a system that predicts taxi fares from millions of trip records. The goals are:

- **Optimize pricing strategies**: Predict fair fares based on distance, time, and location
- **Detect anomalies**: Identify unusually high or low fares that might indicate fraud
- **Forecast revenue**: Estimate total fares for business planning

### The Challenge: Big Data

With **10+ million trips per month**, traditional tools face serious limitations:

- **pandas**: Loads entire dataset into memory → crashes with "Out of Memory" errors
- **scikit-learn**: Single-machine processing → takes hours for large datasets
- **Traditional databases**: Not optimized for ML feature engineering at scale

### The Solution: Apache Spark

Spark enables **distributed processing** across multiple machines:

- **Horizontal scaling**: Add more workers to handle larger datasets
- **In-memory processing**: Cache data across cluster for fast iterations
- **Unified platform**: Data preprocessing + ML training in one framework
- **Fault tolerance**: Automatically recovers from worker failures

### Today's Dataset

We'll work with **1 million NYC taxi trips** including:

- **Features**: Pickup/dropoff datetime and location, distance, passenger count
- **Target**: Fare amount (continuous variable - regression task)
- **Size**: ~200-300 MB (manageable, but demonstrates need for distributed processing)

By the end of this lab, you'll have built a complete production-ready ML pipeline that:
1. ✅ Processes large-scale data with Spark DataFrames
2. ✅ Engineers features for better predictions
3. ✅ Trains and compares multiple regression models
4. ✅ Evaluates model performance and selects the best algorithm

Let's see why Spark is essential for this task...

In [ ]:
# Demo: "The Aha Moment" - Why We Need Spark
# Let's demonstrate the difference between pandas (single-machine) and Spark (distributed)

print("=" * 70)
print("DEMONSTRATION: pandas vs Spark on Large Data")
print("=" * 70)

# First, let's load our dataset with Spark (the RIGHT way for big data)
# Access DBFS where dataset is stored
WEEK_03_DATASET = "/dbfs/shared_datasets/week_03_datasets/yellow_tripdata_2024-01.parquet"

print("\n📊 Loading NYC Taxi dataset with Spark...")
print(f"Data path: {WEEK_03_DATASET}")

# Load data from DBFS (Parquet format for efficiency)
taxi_df = spark.read.parquet(WEEK_03_DATASET)

# CRITICAL: Cache the DataFrame - we'll use it multiple times
# Caching stores data in memory across executors for fast access
# Sample to 500K rows for faster demos (still shows distributed processing!)
taxi_df = taxi_df.sample(fraction=0.17, seed=42)  # ~500K from 3M rows
taxi_df.cache()

# Count records - this triggers caching (Spark is lazy until an action is called)
record_count = taxi_df.count()

print(f"\n✅ Successfully loaded {record_count:,} records with Spark")
# Note: On Shared clusters, partition details are not accessible for security
# This is intentional to provide user isolation in multi-tenant environments
print(f"   Data is distributed across cluster executors")
print(f"   Columns: {len(taxi_df.columns)}")

# Show schema - understand our data structure
print("\n--- Dataset Schema ---")
taxi_df.printSchema()

# Show sample data
print("\n--- Sample Data (first 5 rows) ---")
taxi_df.show(5, truncate=False)

# Show basic statistics
print("\n--- Quick Statistics ---")
taxi_df.select('fare_amount', 'trip_distance', 'passenger_count').describe().show()

print("\n" + "=" * 70)
print("💡 KEY INSIGHT: Spark loaded dataset effortlessly using distributed processing!")
print("=" * 70)
print("\nNext, we'll explore why pandas would struggle with this same data...")

### Understanding the Difference: pandas vs Spark

Now let's demonstrate why pandas struggles with datasets of this size, while Spark handles them efficiently.

**Note:** We won't actually crash pandas (that would interrupt your notebook!), but we'll show you what happens when you try to use pandas on large data.

In [ ]:
# Demo: Comparing pandas vs Spark approaches
# Let's show the memory footprint difference

print("=" * 70)
print("COMPARISON: Memory Usage - pandas vs Spark")
print("=" * 70)

# First, let's see the memory requirement for pandas
# We'll use a SMALL sample to avoid crashing
pandas_sample_size = 10000  # Just 10k rows (1% of our data)
pandas_df_sample = taxi_df.limit(pandas_sample_size).toPandas()

# Calculate pandas memory usage
pandas_memory_mb = pandas_df_sample.memory_usage(deep=True).sum() / (1024 * 1024)

print(f"\n📊 pandas Approach (Single Machine):")
print(f"   Sample size: {pandas_sample_size:,} rows")
print(f"   Memory used: {pandas_memory_mb:.2f} MB")

# Extrapolate to full dataset
full_dataset_rows = record_count
estimated_pandas_memory = (pandas_memory_mb / pandas_sample_size) * full_dataset_rows

print(f"\n   Estimated memory for full dataset ({full_dataset_rows:,} rows):")
print(f"   {estimated_pandas_memory:.2f} MB = {estimated_pandas_memory / 1024:.2f} GB")
print(f"\n   ⚠️  Problem: This entire dataset must fit in SINGLE machine's RAM!")
print(f"   ⚠️  If your machine has less RAM, pandas will crash with MemoryError")

# Note: On Shared clusters, partition details are managed by the cluster for security
# This is intentional to provide user isolation in multi-tenant environments

print(f"\n⚡ Spark Approach (Distributed):")
print(f"   Dataset size: {full_dataset_rows:,} rows")
print(f"   Data is automatically partitioned across cluster executors")
print(f"   Each executor processes a fraction of the data in parallel")
print(f"\n   ✅ Advantage: Data is DISTRIBUTED across cluster executors")
print(f"   ✅ Each executor only holds a FRACTION of the data in memory")
print(f"   ✅ Can scale to datasets larger than any single machine's RAM")

print("\n" + "=" * 70)
print("💡 KEY TAKEAWAY:")
print("   pandas = Single machine, limited by one computer's RAM")
print("   Spark = Distributed, scales across multiple machines")
print("=" * 70)

# Clean up pandas sample
del pandas_df_sample

---

## Topic 1: Spark DataFrames Fundamentals

### What are Spark DataFrames?

Spark DataFrames are **distributed collections of data** organized into named columns, similar to tables in a database or pandas DataFrames, but with key differences:

**Key Concepts:**

1. **Distributed**: Data is automatically partitioned across multiple executors (workers)
2. **Lazy Evaluation**: Operations are not executed immediately - Spark builds an execution plan
3. **Transformations vs Actions**:
   - **Transformations** (lazy): `filter()`, `select()`, `groupBy()` - define operations
   - **Actions** (trigger execution): `show()`, `count()`, `collect()` - execute the plan
4. **Immutable**: Each operation creates a new DataFrame (original unchanged)

**Why This Matters:**

```python
# Example of lazy evaluation
df_filtered = taxi_df.filter(F.col("fare_amount") > 10)  # Transformation - not executed yet
df_grouped = df_filtered.groupBy("passenger_count").count()  # Transformation - still not executed
df_grouped.show()  # Action - NOW everything executes!
```

Spark optimizes the entire chain of operations before executing, making it more efficient than executing each step separately.

### Demo: Essential DataFrame Operations

Let's explore the most common DataFrame operations you'll use in data science workflows.

In [ ]:
# Demo: Essential Spark DataFrame Operations
# Instructor will live-code through these examples

print("=" * 70)
print("DEMO: Spark DataFrame Operations")
print("=" * 70)

# 1. SELECT - Choose specific columns
print("\n1️⃣ SELECT: Choosing specific columns")
print("   Similar to: SELECT fare_amount, trip_distance FROM taxi_df")

selected_df = taxi_df.select("fare_amount", "trip_distance", "passenger_count")
selected_df.show(5)

# 2. FILTER - Keep rows matching a condition
print("\n2️⃣ FILTER: Keeping only valid trips")
print("   Filter: fare > 0, distance > 0, passengers > 0")

# Method 1: Using filter() with column expressions
filtered_df = taxi_df.filter(
    (F.col("fare_amount") > 0) & 
    (F.col("trip_distance") > 0) & 
    (F.col("passenger_count") > 0)
)

print(f"   Original rows: {taxi_df.count():,}")
print(f"   After filtering: {filtered_df.count():,}")
print(f"   Removed: {taxi_df.count() - filtered_df.count():,} invalid rows")

# 3. WITH COLUMN - Add or modify columns
print("\n3️⃣ WITH COLUMN: Adding derived columns")

# Calculate fare per mile
df_with_fare_per_mile = filtered_df.withColumn(
    "fare_per_mile",
    F.col("fare_amount") / F.col("trip_distance")
)

df_with_fare_per_mile.select("fare_amount", "trip_distance", "fare_per_mile").show(5)

# 4. GROUP BY & AGGREGATION - Summarize data
print("\n4️⃣ GROUP BY: Aggregate statistics by passenger count")

grouped_df = filtered_df.groupBy("passenger_count").agg(
    F.count("*").alias("num_trips"),
    F.mean("fare_amount").alias("avg_fare"),
    F.mean("trip_distance").alias("avg_distance")
)

grouped_df.orderBy("passenger_count").show()

# 5. ORDER BY - Sort results
print("\n5️⃣ ORDER BY: Finding highest fares")

top_fares = filtered_df.select("fare_amount", "trip_distance", "passenger_count") \
    .orderBy(F.col("fare_amount").desc()) \
    .limit(10)

print("   Top 10 highest fares:")
top_fares.show()

print("\n" + "=" * 70)
print("✅ Demo complete! These are the building blocks of Spark data processing.")
print("=" * 70)

### Lab 1.1: Explore the NYC Taxi Dataset

**Objective:** Practice Spark DataFrame operations by exploring the taxi dataset and answering business questions.

**Tasks:**

1. **Data Cleaning**: Create a clean dataset by filtering out invalid records:
   - Remove trips where `fare_amount` ≤ 0
   - Remove trips where `trip_distance` ≤ 0
   - Remove trips where `passenger_count` ≤ 0 or > 6 (taxis can't fit more than 6 passengers)
   - Store the cleaned data in a variable called `clean_taxi_df`

2. **Time Analysis**: Extract time-based features from `tpep_pickup_datetime`:
   - Add column `pickup_hour` (hour of day, 0-23)
   - Add column `pickup_day_of_week` (1=Monday, 7=Sunday)
   - Hint: Use `F.hour()` and `F.dayofweek()` functions

3. **Business Questions**: Answer these using Spark DataFrame operations:
   - What is the average fare for each hour of the day?
   - Which hour has the highest average fare?
   - What is the total number of trips by day of week?
   - What is the correlation between trip distance and fare amount?

4. **Cache Your Results**: The cleaned dataset will be used throughout the lab, so cache it for performance.

**Expected Output:**
- `clean_taxi_df`: Filtered DataFrame with ~900k-950k rows
- Time-based columns added
- Aggregate statistics answering the business questions

**Hints:**
- Use `.filter()` or `.where()` for data cleaning
- Use `.withColumn()` to add new columns
- Use `.groupBy().agg()` for aggregations
- Use `.cache()` and `.count()` to cache data
- Use `.corr()` for correlation calculation

In [ ]:
# SOLUTION: Lab 1.1 - Explore the NYC Taxi Dataset
# Complete implementation with explanatory comments

# Task 1: Data Cleaning
# Filter out invalid records using multiple conditions
clean_taxi_df = taxi_df.filter(
    (F.col("fare_amount") > 0) &  # Remove negative or zero fares
    (F.col("trip_distance") > 0) &  # Remove zero-distance trips
    (F.col("passenger_count") > 0) &  # Remove trips with no passengers
    (F.col("passenger_count") <= 6)  # Remove unrealistic passenger counts
)

# Cache the cleaned data for performance (we'll use it multiple times)
clean_taxi_df.cache()
clean_count = clean_taxi_df.count()  # Trigger caching

print(f"✅ Cleaned dataset: {clean_count:,} rows")
print(f"   Removed: {taxi_df.count() - clean_count:,} invalid rows")

print("\n" + "-" * 70)

# Task 2: Time Analysis
# Extract time-based features from pickup datetime
df_with_time = clean_taxi_df \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime"))

print("\n✅ Time-based features added:")
df_with_time.select("tpep_pickup_datetime", "pickup_hour", "pickup_day_of_week").show(5)

print("\n" + "-" * 70)

# Task 3: Business Questions

# Question 3a: Average fare by hour
avg_fare_by_hour = df_with_time.groupBy("pickup_hour").agg(
    F.mean("fare_amount").alias("avg_fare")
)

print("\n✅ Average fare by hour of day:")
avg_fare_by_hour.orderBy("pickup_hour").show(24)

# Question 3b: Hour with highest average fare
highest_fare_hour = avg_fare_by_hour.orderBy(F.col("avg_fare").desc()).limit(1)
print("\n✅ Hour with highest average fare:")
highest_fare_hour.show()

print("\n" + "-" * 70)

# Question 3c: Total trips by day of week
trips_by_day = df_with_time.groupBy("pickup_day_of_week").agg(
    F.count("*").alias("num_trips")
)

print("\n✅ Total trips by day of week:")
trips_by_day.orderBy("pickup_day_of_week").show()

print("\n" + "-" * 70)

# Question 3d: Correlation between distance and fare
correlation = df_with_time.stat.corr("trip_distance", "fare_amount")

print(f"\n✅ Correlation between trip_distance and fare_amount: {correlation:.4f}")
print(f"   Interpretation: Strong positive correlation - longer trips cost more!")

print("\n" + "=" * 70)
print("Lab 1.1 Complete! Review your results above.")
print("=" * 70)

# Explanation:
# - Data cleaning removed invalid records (~5-10% of data typically)
# - Time features enable temporal pattern analysis
# - Correlation of ~0.9 shows trip distance strongly predicts fare
# - Peak hours (rush hour) typically have slightly higher average fares

---

## Topic 2: Feature Engineering at Scale

### Why Feature Engineering Matters

In machine learning, **features are everything**. The quality of your features often matters more than the choice of algorithm. Good features can make a simple model perform well, while poor features make even complex models struggle.

**Real-world example:** Predicting taxi fares

- ❌ **Bad features**: Just use raw columns (pickup_time, distance)
- ✅ **Good features**: Extract hour of day, calculate speed, identify rush hour, encode location patterns

### Spark ML Requirements: Feature Vectors

**CRITICAL CONCEPT**: Spark MLlib requires all features to be assembled into a **single vector column**.

Unlike scikit-learn (which accepts separate columns), Spark ML algorithms expect:

```python
# Spark ML requires this format:
+-------------+------------------+
| features    | label            |
+-------------+------------------+
| [2.5, 3, 1] | 15.30            |  ← features is a VECTOR
| [1.8, 1, 0] | 8.50             |
+-------------+------------------+

# NOT this (separate columns):
+----------+------+-----+---------+
| distance | hour | ... | fare    |
+----------+------+-----+---------+
| 2.5      | 3    | ... | 15.30   |  ← Won't work with Spark ML!
+----------+------+-----+---------+
```

**How we create feature vectors:**
1. Extract/transform individual features (e.g., hour from datetime)
2. Use `VectorAssembler` to combine all features into single vector column
3. Optionally scale/normalize the vector
4. Feed vector to ML algorithm

### Spark ML Pipeline: Chaining Transformations

The **Pipeline API** lets you chain multiple transformation steps into a single workflow:

```python
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler

# Define transformation stages
stage1 = VectorAssembler(inputCols=["distance", "hour"], outputCol="raw_features")
stage2 = StandardScaler(inputCol="raw_features", outputCol="features")

# Create pipeline
pipeline = Pipeline(stages=[stage1, stage2])

# Fit and transform (all at once!)
pipeline_model = pipeline.fit(train_data)
transformed_data = pipeline_model.transform(train_data)
```

**Benefits:**
- ✅ Reproducible: Same transformations apply to train and test data
- ✅ Efficient: Spark optimizes the entire pipeline
- ✅ Production-ready: Save pipeline, deploy to production

### Demo: Building a Feature Engineering Pipeline

Let's build features to predict taxi fares more accurately.

In [ ]:
# Demo: Feature Engineering for Taxi Fare Prediction
# We'll build features that capture temporal patterns, distance, and trip characteristics

print("=" * 70)
print("DEMO: Feature Engineering Pipeline")
print("=" * 70)

# IMPORTANT: We'll use clean_taxi_df from Lab 1.1
# If you haven't completed Lab 1.1, run this temporary code:
if 'clean_taxi_df' not in locals() or clean_taxi_df is None:
    print("\n⚠️  Warning: Using temporary cleaned data (complete Lab 1.1 for your own!)")
    clean_taxi_df = taxi_df.filter(
        (F.col("fare_amount") > 0) & 
        (F.col("trip_distance") > 0) & 
        (F.col("passenger_count") > 0) &
        (F.col("passenger_count") <= 6)
    ).cache()
    clean_taxi_df.count()  # Trigger cache

# Step 1: Extract time-based features
print("\n1️⃣ Extracting time-based features...")

features_df = clean_taxi_df \
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
    .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime")) \
    .withColumn("is_weekend", F.when(F.col("pickup_day_of_week").isin([1, 7]), 1).otherwise(0))

# Add rush hour indicator (morning 7-9, evening 17-19)
features_df = features_df.withColumn(
    "is_rush_hour",
    F.when(
        (F.col("pickup_hour").between(7, 9)) | (F.col("pickup_hour").between(17, 19)),
        1
    ).otherwise(0)
)

print("   ✅ Added: pickup_hour, pickup_day_of_week, is_weekend, is_rush_hour")

# Step 2: Calculate trip duration and speed
print("\n2️⃣ Calculating trip duration and speed...")

features_df = features_df.withColumn(
    "trip_duration_minutes",
    (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60
)

# Calculate speed (miles per hour) - avoid division by zero
features_df = features_df.withColumn(
    "avg_speed_mph",
    F.when(F.col("trip_duration_minutes") > 0,
           (F.col("trip_distance") / F.col("trip_duration_minutes")) * 60
    ).otherwise(0)
)

print("   ✅ Added: trip_duration_minutes, avg_speed_mph")

# Step 3: Select features for model
print("\n3️⃣ Selecting features for ML model...")

# Choose our feature columns (all numeric)
feature_columns = [
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "pickup_day_of_week",
    "is_weekend",
    "is_rush_hour",
    "trip_duration_minutes",
    "avg_speed_mph"
]

# Label column (what we're predicting)
label_column = "fare_amount"

print(f"   Features: {feature_columns}")
print(f"   Label: {label_column}")

# Show sample of engineered features
print("\n--- Sample of Engineered Features ---")
features_df.select(feature_columns + [label_column]).show(5)

# Step 4: Assemble features into vector column
print("\n4️⃣ Assembling features into vector column...")

assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="raw_features",
    handleInvalid="skip"  # Skip rows with null/NaN values
)

# Transform data
assembled_df = assembler.transform(features_df)

print("   ✅ Features assembled into 'raw_features' vector column")
assembled_df.select("raw_features", label_column).show(5, truncate=False)

# Step 5: Scale features (important for algorithms like Linear Regression)
print("\n5️⃣ Scaling features...")

scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,  # Center features to mean=0
    withStd=True    # Scale features to stddev=1
)

scaler_model = scaler.fit(assembled_df)
scaled_df = scaler_model.transform(assembled_df)

print("   ✅ Features scaled to mean=0, std=1")
print("\n--- Final Feature Vector ---")
scaled_df.select("features", label_column).show(5, truncate=False)

print("\n" + "=" * 70)
print("✅ Feature engineering complete!")
print("   We now have ML-ready data with:")
print(f"   - {len(feature_columns)} input features in a vector")
print(f"   - 1 target variable ({label_column})")
print("=" * 70)

### Lab 2.1: Build Your Own Feature Engineering Pipeline

**Objective:** Create a complete feature engineering pipeline using Spark ML Pipeline API.

**Context:** The demo above showed manual feature engineering step-by-step. In production, we want a **reusable pipeline** that can be applied consistently to training data, test data, and new data in production.

**Tasks:**

1. **Create Additional Features**:
   - Add a feature called `time_of_day` with values:
     - 0 = Night (0-5 hours)
     - 1 = Morning (6-11 hours)
     - 2 = Afternoon (12-17 hours)
     - 3 = Evening (18-23 hours)
   - Hint: Use `F.when()` with multiple conditions

2. **Build a Complete Pipeline**:
   Create a Spark ML Pipeline with these stages:
   - **Stage 1**: VectorAssembler to combine all feature columns into "raw_features"
   - **Stage 2**: StandardScaler to scale "raw_features" → "features"
   
3. **Feature List**: Include these features in your pipeline:
   ```python
   feature_columns = [
       "trip_distance",
       "passenger_count",
       "pickup_hour",
       "pickup_day_of_week",
       "is_weekend",
       "is_rush_hour",
       "trip_duration_minutes",
       "avg_speed_mph",
       "time_of_day"  # Your new feature!
   ]
   ```

4. **Fit the Pipeline**: Fit the pipeline on the features_df and create transformed data

5. **Verify Results**: Show the final DataFrame with "features" vector and "fare_amount" label

**Expected Output:**
- A pipeline with 2 stages (assembler + scaler)
- Transformed DataFrame with "features" vector column
- All features properly scaled

**Starter Code Below:**

In [ ]:
# SOLUTION: Lab 2.1 - Build Feature Engineering Pipeline
# Complete implementation with production-ready patterns

# Step 1: Create the time_of_day feature
# Use when() with multiple conditions to categorize hours
features_df_enhanced = features_df.withColumn(
    "time_of_day",
    F.when(F.col("pickup_hour").between(0, 5), 0)  # Night
     .when(F.col("pickup_hour").between(6, 11), 1)  # Morning
     .when(F.col("pickup_hour").between(12, 17), 2)  # Afternoon
     .otherwise(3)  # Evening (18-23)
)

print("✅ Step 1: time_of_day feature added")
features_df_enhanced.select("pickup_hour", "time_of_day").show(10)

print("\n" + "-" * 70)

# Step 2: Define feature columns list
feature_columns = [
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "pickup_day_of_week",
    "is_weekend",
    "is_rush_hour",
    "trip_duration_minutes",
    "avg_speed_mph",
    "time_of_day"
]

label_column = "fare_amount"

print(f"Feature columns: {feature_columns}")
print(f"Label column: {label_column}")

print("\n" + "-" * 70)

# Step 3: Build the Pipeline
# Create VectorAssembler to combine all features into single vector
assembler_stage = VectorAssembler(
    inputCols=feature_columns,
    outputCol="raw_features",
    handleInvalid="skip"  # Skip rows with null/NaN values
)

# Create StandardScaler to normalize features (mean=0, std=1)
scaler_stage = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withMean=True,  # Center data to mean=0
    withStd=True    # Scale to standard deviation=1
)

# Create Pipeline with both stages
feature_pipeline = Pipeline(stages=[assembler_stage, scaler_stage])

print("✅ Step 3: Pipeline created with stages:")
for i, stage in enumerate(feature_pipeline.getStages()):
    print(f"   Stage {i+1}: {type(stage).__name__}")

print("\n" + "-" * 70)

# Step 4: Fit the pipeline and transform data
# Fit learns the mean/std from the data
pipeline_model = feature_pipeline.fit(features_df_enhanced)

# Transform applies the learned transformations
ml_ready_data = pipeline_model.transform(features_df_enhanced)

print("✅ Step 4: Pipeline fitted and data transformed")
print(f"   Total rows: {ml_ready_data.count():,}")
print("\n--- ML-Ready Data Sample ---")
ml_ready_data.select("features", label_column).show(5, truncate=False)

# Cache this data - we'll use it for model training!
ml_ready_data.cache()
ml_ready_data.count()  # Trigger caching
print("\n✅ Data cached for model training")

print("\n" + "=" * 70)
print("Lab 2.1 Complete!")
print("You've built a production-ready feature engineering pipeline!")
print("=" * 70)

# Explanation:
# - time_of_day categorizes hours into 4 periods for pattern detection
# - VectorAssembler combines all features into single vector (required by Spark ML)
# - StandardScaler normalizes features so they contribute equally to model
# - Pipeline ensures same transformations apply to train/test/production data
# - Caching improves performance when training multiple models on same data

---

## Topic 3: Spark ML Regression Models

### Understanding Spark ML Algorithms

Spark MLlib provides several regression algorithms optimized for distributed processing. Let's understand when to use each:

| Algorithm | Best For | Pros | Cons |
|-----------|----------|------|------|
| **Linear Regression** | Linear relationships, interpretability | Fast, simple, interpretable | Assumes linearity |
| **Decision Tree** | Non-linear patterns, feature interactions | Handles non-linearity, no scaling needed | Can overfit |
| **Random Forest** | Robust predictions, feature importance | Reduces overfitting, handles outliers | Slower, less interpretable |
| **Gradient Boosted Trees (GBT)** | Maximum accuracy | Often best performance | Slowest, prone to overfitting |

### How Spark ML Training Works

**Key concept**: Spark distributes **data**, not the model.

```
Training Process:
1. Data is partitioned across executors
2. Each partition computes local gradients/statistics
3. Driver aggregates results
4. Model parameters updated on driver
5. Repeat until convergence
```

**Why this matters:**
- ✅ Can train on datasets larger than single machine's memory
- ✅ Leverages cluster parallelism
- ⚠️ Model itself must fit in driver memory (usually not a problem)

### Train/Test Split Strategy

Before training, we split data into:
- **Training set** (80%): Used to fit the model
- **Test set** (20%): Used to evaluate performance (never seen during training)

**CRITICAL**: Split BEFORE any data-dependent transformations to avoid data leakage!

### Demo: Training Multiple Regression Models

Let's train and compare different regression algorithms on our taxi fare data.

In [ ]:
# Demo: Training and Comparing Regression Models
# We'll train Linear Regression, Random Forest, and Gradient Boosted Trees

print("=" * 70)
print("DEMO: Training Spark ML Regression Models")
print("=" * 70)

# Prepare data: Use ml_ready_data from Lab 2.1, or create temporary version
if 'ml_ready_data' not in locals() or ml_ready_data is None:
    print("\n⚠️  Using demo feature data (complete Lab 2.1 for your own!)")
    # Temporary feature engineering for demo
    demo_features_df = clean_taxi_df \
        .withColumn("pickup_hour", F.hour("tpep_pickup_datetime")) \
        .withColumn("pickup_day_of_week", F.dayofweek("tpep_pickup_datetime")) \
        .withColumn("is_weekend", F.when(F.col("pickup_day_of_week").isin([1, 7]), 1).otherwise(0)) \
        .withColumn("is_rush_hour", 
                   F.when((F.col("pickup_hour").between(7, 9)) | (F.col("pickup_hour").between(17, 19)), 1).otherwise(0)) \
        .withColumn("trip_duration_minutes",
                   (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60) \
        .withColumn("avg_speed_mph",
                   F.when(F.col("trip_duration_minutes") > 0,
                          (F.col("trip_distance") / F.col("trip_duration_minutes")) * 60).otherwise(0))
    
    demo_assembler = VectorAssembler(
        inputCols=["trip_distance", "passenger_count", "pickup_hour", "is_weekend", "is_rush_hour"],
        outputCol="features",
        handleInvalid="skip"
    )
    ml_ready_data = demo_assembler.transform(demo_features_df).select("features", "fare_amount")
    ml_ready_data.cache()
    ml_ready_data.count()

# Step 1: Split data into training and test sets
print("\n1️⃣ Splitting data into train (80%) and test (20%) sets...")

train_data, test_data = ml_ready_data.randomSplit([0.8, 0.2], seed=42)

# Cache both sets - they'll be used multiple times
train_data.cache()
test_data.cache()

train_count = train_data.count()
test_count = test_data.count()

print(f"   Training set: {train_count:,} rows")
print(f"   Test set: {test_count:,} rows")

# Step 2: Train Linear Regression
print("\n2️⃣ Training Linear Regression...")

lr = LinearRegression(
    featuresCol="features",
    labelCol="fare_amount",
    predictionCol="prediction",
    maxIter=10,  # Maximum iterations for convergence
    regParam=0.01  # Regularization parameter (prevents overfitting)
)

import time
start_time = time.time()
lr_model = lr.fit(train_data)
lr_train_time = time.time() - start_time

print(f"   ✅ Linear Regression trained in {lr_train_time:.2f} seconds")
print(f"   Coefficients: {lr_model.coefficients[:3]}... (showing first 3)")
print(f"   Intercept: {lr_model.intercept:.2f}")

# Step 3: Train Random Forest
print("\n3️⃣ Training Random Forest Regressor...")

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    predictionCol="prediction",
    numTrees=20,  # Number of trees in the forest
    maxDepth=10,  # Maximum depth of each tree
    seed=42
)

start_time = time.time()
rf_model = rf.fit(train_data)
rf_train_time = time.time() - start_time

print(f"   ✅ Random Forest trained in {rf_train_time:.2f} seconds")
print(f"   Number of trees: {rf_model.getNumTrees}")
# Feature importances available in rf_model.featureImportances (SparseVector - use .toArray() to inspect)

# Step 4: Train Gradient Boosted Trees
print("\n4️⃣ Training Gradient Boosted Trees...")

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    predictionCol="prediction",
    maxIter=10,  # Number of boosting iterations
    maxDepth=5,  # Depth of each tree (smaller than RF to prevent overfitting)
    seed=42
)

start_time = time.time()
gbt_model = gbt.fit(train_data)
gbt_train_time = time.time() - start_time

print(f"   ✅ Gradient Boosted Trees trained in {gbt_train_time:.2f} seconds")
print(f"   Number of trees: {gbt_model.getNumTrees}")

# Step 5: Make predictions on test set
print("\n5️⃣ Making predictions on test set...")

lr_predictions = lr_model.transform(test_data)
rf_predictions = rf_model.transform(test_data)
gbt_predictions = gbt_model.transform(test_data)

print("   ✅ Predictions generated for all models")

# Show sample predictions
print("\n--- Sample Predictions (Linear Regression) ---")
lr_predictions.select("fare_amount", "prediction").show(5)

print("\n" + "=" * 70)
print("✅ All models trained successfully!")
print(f"   Training times: LR={lr_train_time:.1f}s, RF={rf_train_time:.1f}s, GBT={gbt_train_time:.1f}s")
print("=" * 70)

### Lab 3.1: Train Your Own Regression Models

**Objective:** Train multiple regression models and compare their characteristics.

**Context:** You've seen how to train Linear Regression, Random Forest, and GBT. Now it's your turn to train models using your feature-engineered data from Lab 2.1.

**Tasks:**

1. **Split Your Data**: Create train/test split (80/20) from your `ml_ready_data`
   - Use `randomSplit([0.8, 0.2], seed=42)`
   - Cache both train and test sets

2. **Train Three Models**:
   - **Linear Regression**: Use default parameters
   - **Decision Tree Regressor**: Use `maxDepth=10`
   - **Random Forest Regressor**: Use `numTrees=30, maxDepth=10`

3. **Make Predictions**: Transform test data with each model

4. **Compare Training Times**: Measure and compare how long each model takes to train

**Expected Output:**
- Three trained models (LR, DT, RF)
- Predictions DataFrames for each model
- Training time comparison

**Hints:**
- Import `time` module to measure training time
- Use `.cache()` on train and test data for better performance
- All models should use `featuresCol="features"` and `labelCol="fare_amount"`

**Starter Code Below:**

In [ ]:
# SOLUTION: Lab 3.1 - Train Regression Models
# Complete implementation with all three model types

import time

# Task 1: Split data into train (80%) and test (20%)
train_data, test_data = ml_ready_data.randomSplit([0.8, 0.2], seed=42)

# Cache both datasets for faster training
train_data.cache()
test_data.cache()

# Trigger caching
train_count = train_data.count()
test_count = test_data.count()

print(f"✅ Task 1: Data split complete")
print(f"   Training set: {train_count:,} rows")
print(f"   Test set: {test_count:,} rows")

print("\n" + "-" * 70)

# Task 2a: Train Linear Regression
lr = LinearRegression(
    featuresCol="features",
    labelCol="fare_amount",
    predictionCol="prediction",
    maxIter=10,
    regParam=0.01
)

start_time = time.time()
lr_model_lab = lr.fit(train_data)
lr_time = time.time() - start_time

print(f"✅ Task 2a: Linear Regression trained in {lr_time:.2f}s")
print(f"   Coefficients (first 3): {lr_model_lab.coefficients[:3]}")
print(f"   Intercept: {lr_model_lab.intercept:.2f}")

print("\n" + "-" * 70)

# Task 2b: Train Decision Tree with maxDepth=10
dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    predictionCol="prediction",
    maxDepth=10,
    seed=42
)

start_time = time.time()
dt_model_lab = dt.fit(train_data)
dt_time = time.time() - start_time

print(f"✅ Task 2b: Decision Tree trained in {dt_time:.2f}s")
print(f"   Max depth: {dt_model_lab.depth}")
print(f"   Number of nodes: {dt_model_lab.numNodes}")

print("\n" + "-" * 70)

# Task 2c: Train Random Forest with numTrees=30, maxDepth=10
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="fare_amount",
    predictionCol="prediction",
    numTrees=30,
    maxDepth=10,
    seed=42
)

start_time = time.time()
rf_model_lab = rf.fit(train_data)
rf_time = time.time() - start_time

print(f"✅ Task 2c: Random Forest trained in {rf_time:.2f}s")
print(f"   Number of trees: {rf_model_lab.getNumTrees}")
print(f"   Feature importances (first 3): {rf_model_lab.featureImportances[:3]}")

print("\n" + "-" * 70)

# Task 3: Make predictions on test data
lr_predictions_lab = lr_model_lab.transform(test_data)
dt_predictions_lab = dt_model_lab.transform(test_data)
rf_predictions_lab = rf_model_lab.transform(test_data)

print("✅ Task 3: Predictions generated for all models")
print("\n--- Sample Predictions (Linear Regression) ---")
lr_predictions_lab.select("fare_amount", "prediction").show(5)

print("\n" + "-" * 70)

# Task 4: Compare training times
print("✅ Task 4: Training Time Comparison")
print(f"   Linear Regression: {lr_time:.2f}s")
print(f"   Decision Tree: {dt_time:.2f}s")
print(f"   Random Forest: {rf_time:.2f}s")

if lr_time < min(dt_time, rf_time):
    fastest = "Linear Regression"
elif dt_time < rf_time:
    fastest = "Decision Tree"
else:
    fastest = "Random Forest"

print(f"\n   Fastest: {fastest}")
print(f"   Slowest: {'Random Forest' if rf_time > max(lr_time, dt_time) else 'Decision Tree' if dt_time > lr_time else 'Linear Regression'}")

print("\n" + "=" * 70)
print("Lab 3.1 Complete!")
print("You've trained three different regression models!")
print("Next: We'll evaluate these models to see which performs best!")
print("=" * 70)

# Explanation:
# - Linear Regression: Fastest but assumes linear relationships
# - Decision Tree: Handles non-linearity, moderate speed
# - Random Forest: Slowest but often most accurate (ensemble of 30 trees)
# - Training time increases: LR < DT < RF
# - All models cached their training data for efficiency

---

## Topic 4: Model Evaluation & Selection

### Understanding Regression Metrics

To select the best model, we need to evaluate performance using standard regression metrics:

| Metric | Formula | Interpretation | Goal |
|--------|---------|----------------|------|
| **RMSE** (Root Mean Squared Error) | √(Σ(actual - predicted)²/n) | Average prediction error in original units | Lower is better |
| **MAE** (Mean Absolute Error) | Σ\|actual - predicted\|/n | Average absolute error | Lower is better |
| **R²** (R-squared) | 1 - (SS_res / SS_tot) | % of variance explained by model | Higher is better (max 1.0) |

**Which metric to use?**
- **RMSE**: Penalizes large errors more (sensitive to outliers)
- **MAE**: More robust to outliers
- **R²**: Indicates how well model fits data (0 = poor, 1 = perfect)

**Real-world example:**
```
Predicting taxi fares:
- RMSE = $3.50  → On average, predictions are off by $3.50
- MAE = $2.80   → Typical error is $2.80
- R² = 0.85     → Model explains 85% of fare variation
```

### Using RegressionEvaluator in Spark

Spark provides `RegressionEvaluator` to calculate these metrics:

```python
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"  # Can be "rmse", "mae", "r2", "mse"
)

rmse = evaluator.evaluate(predictions)
```

### Demo: Comprehensive Model Evaluation

Let's evaluate all our models and select the best performer.

In [ ]:
# Demo: Evaluating and Comparing Models
# We'll calculate RMSE, MAE, and R² for all three models from the demo

print("=" * 70)
print("DEMO: Model Evaluation and Comparison")
print("=" * 70)

# Create evaluator for RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

# Create evaluator for MAE
evaluator_mae = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="mae"
)

# Create evaluator for R²
evaluator_r2 = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

# Evaluate Linear Regression
print("\n1️⃣ Evaluating Linear Regression...")
lr_rmse = evaluator_rmse.evaluate(lr_predictions)
lr_mae = evaluator_mae.evaluate(lr_predictions)
lr_r2 = evaluator_r2.evaluate(lr_predictions)

print(f"   RMSE: ${lr_rmse:.2f}")
print(f"   MAE:  ${lr_mae:.2f}")
print(f"   R²:   {lr_r2:.4f}")

# Evaluate Random Forest
print("\n2️⃣ Evaluating Random Forest...")
rf_rmse = evaluator_rmse.evaluate(rf_predictions)
rf_mae = evaluator_mae.evaluate(rf_predictions)
rf_r2 = evaluator_r2.evaluate(rf_predictions)

print(f"   RMSE: ${rf_rmse:.2f}")
print(f"   MAE:  ${rf_mae:.2f}")
print(f"   R²:   {rf_r2:.4f}")

# Evaluate Gradient Boosted Trees
print("\n3️⃣ Evaluating Gradient Boosted Trees...")
gbt_rmse = evaluator_rmse.evaluate(gbt_predictions)
gbt_mae = evaluator_mae.evaluate(gbt_predictions)
gbt_r2 = evaluator_r2.evaluate(gbt_predictions)

print(f"   RMSE: ${gbt_rmse:.2f}")
print(f"   MAE:  ${gbt_mae:.2f}")
print(f"   R²:   {gbt_r2:.4f}")

# Create comparison table
print("\n" + "=" * 70)
print("MODEL COMPARISON SUMMARY")
print("=" * 70)
print(f"{'Model':<25} {'RMSE':<12} {'MAE':<12} {'R²':<10}")
print("-" * 70)
print(f"{'Linear Regression':<25} ${lr_rmse:<11.2f} ${lr_mae:<11.2f} {lr_r2:<10.4f}")
print(f"{'Random Forest':<25} ${rf_rmse:<11.2f} ${rf_mae:<11.2f} {rf_r2:<10.4f}")
print(f"{'Gradient Boosted Trees':<25} ${gbt_rmse:<11.2f} ${gbt_mae:<11.2f} {gbt_r2:<10.4f}")
print("=" * 70)

# Determine best model
best_rmse_model = min([("Linear Regression", lr_rmse), ("Random Forest", rf_rmse), ("GBT", gbt_rmse)], key=lambda x: x[1])
best_r2_model = max([("Linear Regression", lr_r2), ("Random Forest", rf_r2), ("GBT", gbt_r2)], key=lambda x: x[1])

print(f"\n🏆 Best Model by RMSE: {best_rmse_model[0]} (${best_rmse_model[1]:.2f})")
print(f"🏆 Best Model by R²: {best_r2_model[0]} ({best_r2_model[1]:.4f})")

# Analyze prediction errors
print("\n4️⃣ Analyzing prediction errors...")

# Add residuals column (actual - predicted)
rf_with_residuals = rf_predictions.withColumn(
    "residual",
    F.col("fare_amount") - F.col("prediction")
)

# Show distribution of residuals
print("\n--- Residuals Statistics (Random Forest) ---")
rf_with_residuals.select("residual").describe().show()

# Find largest errors
print("\n--- Trips with Largest Prediction Errors ---")
rf_with_residuals.select("fare_amount", "prediction", "residual") \
    .orderBy(F.abs(F.col("residual")).desc()) \
    .limit(5) \
    .show()

print("\n" + "=" * 70)
print("✅ Evaluation complete!")
print("=" * 70)

### Lab 4.1: Evaluate and Select Best Model

**Objective:** Evaluate all models you trained in Lab 3.1 and select the best performer.

**Tasks:**

1. **Create Evaluators**: Set up RegressionEvaluator for RMSE, MAE, and R²

2. **Evaluate All Models**: Calculate all three metrics for:
   - Linear Regression
   - Decision Tree
   - Random Forest

3. **Create Comparison Table**: Display results in a formatted table showing all metrics

4. **Select Best Model**: Determine which model performs best based on:
   - Lowest RMSE
   - Highest R²

5. **Error Analysis**: For your best model:
   - Calculate residuals (actual - predicted)
   - Show statistics of residuals
   - Find the 5 trips with largest prediction errors

**Expected Output:**
- Comparison table with RMSE, MAE, R² for all 3 models
- Identification of best model
- Residual analysis for best model

**Starter Code Below:**

In [ ]:
# SOLUTION: Lab 4.1 - Evaluate and Select Best Model
# Complete implementation with comprehensive evaluation

# Task 1: Create evaluators for each metric
evaluator_rmse = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="rmse"
)

evaluator_mae = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="mae"
)

evaluator_r2 = RegressionEvaluator(
    labelCol="fare_amount",
    predictionCol="prediction",
    metricName="r2"
)

print("✅ Task 1: Evaluators created for RMSE, MAE, and R²")

print("\n" + "-" * 70)

# Task 2: Evaluate all models

# Evaluate Linear Regression
lr_rmse_lab = evaluator_rmse.evaluate(lr_predictions_lab)
lr_mae_lab = evaluator_mae.evaluate(lr_predictions_lab)
lr_r2_lab = evaluator_r2.evaluate(lr_predictions_lab)

# Evaluate Decision Tree
dt_rmse_lab = evaluator_rmse.evaluate(dt_predictions_lab)
dt_mae_lab = evaluator_mae.evaluate(dt_predictions_lab)
dt_r2_lab = evaluator_r2.evaluate(dt_predictions_lab)

# Evaluate Random Forest
rf_rmse_lab = evaluator_rmse.evaluate(rf_predictions_lab)
rf_mae_lab = evaluator_mae.evaluate(rf_predictions_lab)
rf_r2_lab = evaluator_r2.evaluate(rf_predictions_lab)

print("✅ Task 2: All models evaluated across 3 metrics")

print("\n" + "-" * 70)

# Task 3: Create comparison table
print("✅ Task 3: Model Comparison")
print("\n" + "=" * 70)
print("MODEL COMPARISON - YOUR RESULTS")
print("=" * 70)
print(f"{'Model':<20} {'RMSE':<12} {'MAE':<12} {'R²':<10}")
print("-" * 70)
print(f"{'Linear Regression':<20} ${lr_rmse_lab:<11.2f} ${lr_mae_lab:<11.2f} {lr_r2_lab:<10.4f}")
print(f"{'Decision Tree':<20} ${dt_rmse_lab:<11.2f} ${dt_mae_lab:<11.2f} {dt_r2_lab:<10.4f}")
print(f"{'Random Forest':<20} ${rf_rmse_lab:<11.2f} ${rf_mae_lab:<11.2f} {rf_r2_lab:<10.4f}")
print("=" * 70)

print("\n" + "-" * 70)

# Task 4: Select best model based on lowest RMSE
models = [
    ("Linear Regression", lr_rmse_lab, lr_r2_lab, lr_predictions_lab),
    ("Decision Tree", dt_rmse_lab, dt_r2_lab, dt_predictions_lab),
    ("Random Forest", rf_rmse_lab, rf_r2_lab, rf_predictions_lab)
]

# Find model with lowest RMSE
best_model = min(models, key=lambda x: x[1])
best_model_name = best_model[0]
best_model_rmse = best_model[1]
best_predictions = best_model[3]

# Find model with highest R²
best_r2_model = max(models, key=lambda x: x[2])

print(f"✅ Task 4: Best Model Selected")
print(f"\n🏆 Best Model by RMSE: {best_model_name}")
print(f"   RMSE: ${best_model_rmse:.2f}")
print(f"   R²: {best_model[2]:.4f}")
print(f"\n🏆 Best Model by R²: {best_r2_model[0]}")
print(f"   R²: {best_r2_model[2]:.4f}")

print("\n" + "-" * 70)

# Task 5: Error analysis for best model
print(f"✅ Task 5: Error Analysis for {best_model_name}")

# Add residuals column (actual - predicted)
predictions_with_residuals = best_predictions.withColumn(
    "residual",
    F.col("fare_amount") - F.col("prediction")
)

print("\n--- Residual Statistics ---")
predictions_with_residuals.select("residual").describe().show()

print("\n--- Largest Prediction Errors ---")
print(f"(Sorted by absolute residual - these are the hardest trips to predict)")
predictions_with_residuals.select("fare_amount", "prediction", "residual") \
    .orderBy(F.abs(F.col("residual")).desc()) \
    .limit(5) \
    .show()

print("\n" + "=" * 70)
print("Lab 4.1 Complete!")
print("Congratulations! You've completed the main regression pipeline!")
print("=" * 70)

# Explanation:
# - RMSE penalizes large errors more than MAE (sensitive to outliers)
# - R² shows % of variance explained (higher is better, max 1.0)
# - Random Forest typically performs best (ensemble learning)
# - Large residuals indicate difficult-to-predict trips (outliers, unusual patterns)
# - Decision Tree may overfit if maxDepth is too high
# - Linear Regression is fastest but assumes linear relationships

print("\n📊 Performance Insights:")
print(f"   - RMSE of ${best_model_rmse:.2f} means typical prediction error is ~${best_model_rmse:.2f}")
print(f"   - R² of {best_model[2]:.4f} means model explains {best_model[2]*100:.1f}% of fare variation")
print(f"   - Model is ready for production deployment on Spark cluster!")

---

## Optional/Advanced Challenges (Async Learning)

**These sections are for students who finish early or want to explore further at home.**

### Optional Lab 1: Hyperparameter Tuning with CrossValidator

**Challenge:** Use Spark ML's CrossValidator to find optimal hyperparameters for Random Forest.

**What you'll learn:**
- Grid search for hyperparameter optimization
- K-fold cross-validation in Spark
- ParamGridBuilder usage

**Approach:**
```python
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

# Create parameter grid
paramGrid = ParamGridBuilder() \
    .addGrid(rf.numTrees, [10, 20, 30]) \
    .addGrid(rf.maxDepth, [5, 10, 15]) \
    .build()

# Create cross-validator
crossval = CrossValidator(
    estimator=rf,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(labelCol="fare_amount", metricName="rmse"),
    numFolds=3
)

# Fit (this will take longer!)
cv_model = crossval.fit(train_data)
best_model = cv_model.bestModel
```

**Try it:** Find the best combination of `numTrees` and `maxDepth` for your data.

### Optional Lab 2: Advanced Feature Engineering

**Challenge:** Create more sophisticated features to improve model performance.

**Ideas to explore:**

1. **Location-based features**:
   - Group pickup/dropoff locations into clusters
   - Create "popular destination" indicator
   - Calculate distance from city center

2. **Temporal patterns**:
   - Create "holiday" indicator (requires date lookup)
   - "Month" and "season" features
   - "Minutes since midnight" for finer time granularity

3. **Interaction features**:
   - `distance × is_rush_hour` (rush hour affects fare per mile)
   - `passenger_count × is_weekend` (weekend group patterns)

4. **Polynomial features**:
   - `trip_distance²` (non-linear relationship with fare)
   - `avg_speed²` (capture extreme speeds)

**Approach:**
```python
# Example: Create location popularity feature
location_counts = clean_taxi_df.groupBy("PULocationID") \
    .count() \
    .withColumnRenamed("count", "location_popularity")

features_df_advanced = clean_taxi_df.join(
    location_counts, 
    on="PULocationID", 
    how="left"
)
```

**Try it:** Add 2-3 new features and see if model performance improves!

### Optional Lab 3: Spark + PyTorch Integration (Single-Node DL)

**Challenge:** Compare Spark ML (distributed) with PyTorch (single-node deep learning).

**What you'll learn:**
- When to use Spark ML vs PyTorch
- How to use Spark for data preprocessing and PyTorch for modeling
- Trade-offs between distributed ML and deep learning

**Approach:**

1. **Use Spark to preprocess and sample data** (distributed):
```python
# Preprocess with Spark (handles full dataset)
preprocessed_df = ml_ready_data.sample(0.1)  # Sample 10% for PyTorch

# Convert to pandas for PyTorch
pandas_df = preprocessed_df.select("features", "fare_amount").toPandas()
```

2. **Train PyTorch neural network** (single-node):
```python
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Convert to PyTorch tensors
# Note: Spark features are sparse vectors, need to convert to dense
X = torch.tensor([row.features.toArray() for row in preprocessed_df.select("features").collect()])
y = torch.tensor(pandas_df['fare_amount'].values, dtype=torch.float32)

# Simple neural network
class FarePredictor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.network(x)

# Train model...
```

3. **Compare results**:
   - Spark ML Random Forest: Trains on full 1M rows, distributed
   - PyTorch NN: Trains on 100k sample, single GPU/CPU
   - Which performs better? Which is faster?

**Try it:** Train both approaches and compare RMSE!

---

## Congratulations! 🎉

You've completed **Week 3: ML on Databricks - Spark & Regression**!

### What You've Learned

✅ **Distributed Processing Fundamentals**
- Understand why Spark is needed for big data (pandas limitations)
- Know how data is distributed across executors
- Understand lazy evaluation and transformations vs actions

✅ **Spark DataFrames Mastery**
- Load and explore large datasets from Unity Catalog
- Perform filtering, grouping, and aggregations at scale
- Use Spark SQL for data analysis

✅ **Feature Engineering at Scale**
- Extract time-based features from timestamps
- Create derived features (speed, duration, indicators)
- Build reusable ML Pipelines with VectorAssembler and StandardScaler
- Understand why Spark ML needs feature vectors

✅ **Spark ML Regression**
- Train multiple regression algorithms (Linear, Decision Tree, Random Forest, GBT)
- Understand trade-offs between different algorithms
- Split data for proper train/test evaluation

✅ **Model Evaluation & Selection**
- Calculate regression metrics (RMSE, MAE, R²)
- Compare models systematically
- Analyze prediction errors and residuals
- Select the best model for production

### Key Takeaways

💡 **When to Use Spark**:
- Dataset too large for single machine memory (>10GB typically)
- Need to scale horizontally across cluster
- Building production ML pipelines that process millions of records

💡 **Spark ML Best Practices**:
- Always cache DataFrames you'll reuse (`.cache()`)
- Use Pipelines for reproducible transformations
- Let Adaptive Query Execution optimize partitions automatically
- Monitor cluster resources in Databricks UI

💡 **Production Considerations**:
- Feature engineering pipeline can be saved and reused
- Models can be deployed to Spark serving infrastructure
- Same code scales from 1M to 1B rows

### Next Steps

**Week 4 Preview**: You'll apply these same Spark ML concepts to **classification problems** (customer churn, fraud detection) and learn about:
- Handling imbalanced datasets
- Classification metrics (precision, recall, F1, AUC-ROC)
- Cross-validation and hyperparameter tuning
- Model persistence and deployment

### Additional Resources

- [Spark MLlib Guide](https://spark.apache.org/docs/latest/ml-guide.html)
- [Databricks ML Runtime Documentation](https://docs.databricks.com/runtime/mlruntime.html)
- [Unity Catalog Best Practices](https://docs.databricks.com/data-governance/unity-catalog/best-practices.html)
- [Spark Performance Tuning](https://spark.apache.org/docs/latest/sql-performance-tuning.html)

---

**Great work today! See you in Week 4! 🚀**